# 01 Data loading

Load the two training sets from TDC: `CYP3A4_Veith` for Model A (does a compound inhibit CYP3A4) and `hERG` for Model B (does a compound block the hERG channel). Check the size and the label balance of each, because the hERG set is small and uneven and that decides how we have to validate Model B.

Then convert both to Morgan fingerprints with `src/features.py` and save them, so the later notebooks do not have to featurise again. This notebook runs end to end.

In [ ]:
import sys
import os

# Let the notebook import the helpers that live in src/.
sys.path.append(os.path.abspath(os.path.join("..", "src")))

import numpy as np
import pandas as pd
import features

RAW_DIR = os.path.join("..", "data", "raw")
PROCESSED_DIR = os.path.join("..", "data", "processed")
print("fingerprint setting: radius", features.FP_RADIUS, "size", features.FP_SIZE)

## Model A data: CYP3A4 inhibition

In [ ]:
from tdc.single_pred import ADME

# About 12k compounds from the Veith high throughput inhibition assay.
cyp_data = ADME(name="CYP3A4_Veith", path=RAW_DIR)
cyp_df = cyp_data.get_data()

print("CYP3A4_Veith shape:", cyp_df.shape)
print("columns:", list(cyp_df.columns))
cyp_df.head()

In [ ]:
# Label balance decides whether plain accuracy would be misleading.
print("CYP3A4 label counts:")
print(cyp_df["Y"].value_counts())
print("positive fraction:", round(float(cyp_df["Y"].mean()), 3))

## Model B data: hERG block

In [ ]:
from tdc.single_pred import Tox

# Only a few hundred compounds, which is why Model B needs a scaffold split
# or cross validation rather than one random holdout.
herg_data = Tox(name="hERG", path=RAW_DIR)
herg_df = herg_data.get_data()

print("hERG shape:", herg_df.shape)
print("columns:", list(herg_df.columns))
print("hERG label counts:")
print(herg_df["Y"].value_counts())
print("positive fraction:", round(float(herg_df["Y"].mean()), 3))

## Fingerprints

Some SMILES in public sets fail to parse, so `smiles_list_to_matrix` also returns the row indices that survived. We use those indices to keep the labels lined up with the rows.

In [ ]:
X_cyp, kept_cyp = features.smiles_list_to_matrix(cyp_df["Drug"].tolist())
y_cyp = cyp_df["Y"].values[kept_cyp]

print("CYP fingerprints:", X_cyp.shape)
print("dropped as unparseable:", len(cyp_df) - len(kept_cyp))
print("mean bits set per molecule:", round(float(X_cyp.sum(axis=1).mean()), 1))

In [ ]:
X_herg, kept_herg = features.smiles_list_to_matrix(herg_df["Drug"].tolist())
y_herg = herg_df["Y"].values[kept_herg]

print("hERG fingerprints:", X_herg.shape)
print("dropped as unparseable:", len(herg_df) - len(kept_herg))
print("mean bits set per molecule:", round(float(X_herg.sum(axis=1).mean()), 1))

In [ ]:
# Save so notebooks 02 and 03 can start straight from the matrices.
np.save(os.path.join(PROCESSED_DIR, "X_cyp3a4.npy"), X_cyp)
np.save(os.path.join(PROCESSED_DIR, "y_cyp3a4.npy"), y_cyp)
np.save(os.path.join(PROCESSED_DIR, "X_herg.npy"), X_herg)
np.save(os.path.join(PROCESSED_DIR, "y_herg.npy"), y_herg)

# Keep the SMILES too. SHAP and the fragment mapping need the molecules back.
cyp_df.iloc[kept_cyp].to_csv(os.path.join(PROCESSED_DIR, "cyp3a4_clean.csv"), index=False)
herg_df.iloc[kept_herg].to_csv(os.path.join(PROCESSED_DIR, "herg_clean.csv"), index=False)
print("saved to", PROCESSED_DIR)